In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
%pip install ragas

Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for ragas from https://files.pythonhosted.org/packages/4d/e0/1fecd22c93d3ed66453cbbdefd05528331af4d33b2b76a370d751231912c/ragas-0.4.3-py3-none-any.whl.metadata
  Using cached ragas-0.4.3-py3-none-any.whl.metadata (23 kB)
  Obtaining dependency information for datasets>=4.0.0 from https://files.pythonhosted.org/packages/05/66/73034ad30b59f13439b75e620989dacba4c047256e358ba7c2e9ec98ea22/datasets-5.0.0-py3-none-any.whl.metadata
  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Obtaining dependency information for appdirs from https://files.pythonhosted.org/packages/3b/00/2344469e2084fb287c2e0b57b72910309874c3245463acd6cf5e3db69324/appdirs-1.4.4-py2.py3-none-any.whl.metadata
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Obtaining dependency information for diskcache>=5.6.3 from https://files.pythonhosted.org/packages/3f/27/4570e78fc0bf5ea0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.83.13 requires openai==2.24.0, but you have openai 2.44.0 which is incompatible.

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Context precision - Experiment

In [5]:
from langchain_groq import ChatGroq
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import ContextPrecision  # <-- old API, not collections
from ragas import SingleTurnSample
import os
groq_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

llm = LangchainLLMWrapper(groq_llm)

scorer = ContextPrecision(llm=llm)

sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    reference="The Eiffel Tower is located in Paris.",
    retrieved_contexts=[
        "The Eiffel Tower is located in Paris."
        "The Brandenburg Gate is located in Berlin.",
        
    ]
)

result = await scorer.single_turn_ascore(sample)
print(f"Context Precision Score: {result}")

C:\Users\Dell\AppData\Local\Temp\ipykernel_23176\91799341.py:3: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextPrecision  # <-- old API, not collections
C:\Users\Dell\AppData\Local\Temp\ipykernel_23176\91799341.py:12: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  llm = LangchainLLMWrapper(groq_llm)


Context Precision Score: 0.9999999999


Context Recall - Experiment

In [15]:
from langchain_groq import ChatGroq
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import ContextRecall  # <-- old API, not collections
from ragas import SingleTurnSample
import os
groq_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

llm = LangchainLLMWrapper(groq_llm)


# Create metric
scorer = ContextRecall(llm=llm)

# Evaluate
sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located and what is its height?",
    reference="The Eiffel Tower is in Paris and is 330 meters tall.",
    retrieved_contexts=["The Eiffel Tower is located in Paris."]  # missing height
)
result = await scorer.single_turn_ascore(sample)
print(f"Context Recall Score: {result}")

C:\Users\Dell\AppData\Local\Temp\ipykernel_23176\3117525549.py:3: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import ContextRecall  # <-- old API, not collections


C:\Users\Dell\AppData\Local\Temp\ipykernel_23176\3117525549.py:12: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  llm = LangchainLLMWrapper(groq_llm)


Context Recall Score: 0.5


Answer Relevancy - Experiment

In [18]:
%pip install langchain-huggingface sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import AnswerRelevancy
from ragas import SingleTurnSample
import os

# Patch ChatGroq to force n=1
class GroqWrapper(ChatGroq):
    def _create_chat_result(self, response, generation_info=None):
        return super()._create_chat_result(response, generation_info)
    
    def bind(self, **kwargs):
        kwargs.pop("n", None)  # remove n if passed
        return super().bind(**kwargs)

groq_llm = GroqWrapper(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)
llm = LangchainLLMWrapper(groq_llm)

hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

# Override n in the metric itself
scorer = AnswerRelevancy(llm=llm, embeddings=embeddings)
scorer.strictness = 1  # controls how many questions are generated (default 3, uses n=3)

sample = SingleTurnSample(
    user_input="When was the first super bowl?",
    response="The first superbowl was held on Jan 15, 1968"
)

result = await scorer.single_turn_ascore(sample)
print(f"Answer Relevancy Score: {result}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2278.85it/s]


Answer Relevancy Score: 0.9567587630881884


In [23]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import Faithfulness
from ragas import SingleTurnSample
import os

# Patch ChatGroq to force n=1
class GroqWrapper(ChatGroq):
    def _create_chat_result(self, response, generation_info=None):
        return super()._create_chat_result(response, generation_info)
    
    def bind(self, **kwargs):
        kwargs.pop("n", None)  # remove n if passed
        return super().bind(**kwargs)

groq_llm = GroqWrapper(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)
llm = LangchainLLMWrapper(groq_llm)

hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

# Override n in the metric itself
scorer = Faithfulness(llm=llm)
scorer.strictness = 1  # controls how many questions are generated (default 3, uses n=3)

sample = SingleTurnSample(
    user_input="When was the first super bowl?",
    response="The first superbowl was held on Jan 15, 1968",
    retrieved_contexts=[
        "The First AFL–NFL World Championship Game was an American football game played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles."
    ]
)

result = await scorer.single_turn_ascore(sample)
print(f"Answer Relevancy Score: {result}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1973.81it/s]


Answer Relevancy Score: 0.0
